In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from sklearn.datasets import load_iris, fetch_openml

%matplotlib widget

# Data Loading

In [2]:
# Loading data
filename = 'data/data9.txt'
df_raw = pd.read_csv(filename, sep='\t', header=None)
df_raw = df_raw.dropna(axis=1, how='all')
X = df_raw.values
print(f"Loaded Data: {X.shape[0]} rows, {X.shape[1]} columns")

Loaded Data: 220 rows, 16 columns


In [3]:
iris = load_iris()
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_mnist = mnist.data
y_mnist = mnist.target
y_iris = iris.target
df_mnist = pd.DataFrame(X_mnist).to_numpy()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names).to_numpy()

In [ ]:
# PCA Auxiliary functions
def pca(X, pca_component:int = 3):
    pca = PCA(n_components=pca_component)
    pca_X = pca.fit_transform(X)
    return pca_X

def pca_plot(pca_X, y = None, title = 'Y', plot_2d:bool = True, plot_3d:bool = True, figsize = (8, 8)):
    plt.figure(figsize=figsize)
    if y is None:
        y = np.zeros(pca_X.shape[0])
    
    if plot_2d:
        plt.scatter(pca_X[:, 0], pca_X[:, 1], s=1, c=y, cmap='viridis', alpha=0.7)
        plt.xlabel('Principal Component 1')
        plt.ylabel('Principal Component 2')
        plt.colorbar(label='Cluster Label')
        plt.title(title + '(dim = 2)')
        plt.show()
    
    if plot_3d:
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        ax.scatter(pca_X[:, 0], pca_X[:, 1], pca_X[:, 2], c=y, cmap='viridis', alpha=0.7)
        ax.set_xlabel('Principal Component 1')
        ax.set_ylabel('Principal Component 2')
        ax.set_zlabel('Principal Component 3')
        ax.set_title(title + '(dim = 3)')
        plt.show()

def kmeans_pca(X, pca_component:int = 3, k:int = 6):
    pca = PCA(n_components=pca_component)
    pca_X = pca.fit_transform(X)
    kmeans_X = KMeans(n_clusters=k, random_state=42)
    cluster_labels = kmeans_X.fit_predict(X)
    return pca_X, cluster_labels

def kmeans_pca_plot(X, y_true = None, y_pred = None, pca_component:int = 3, plot_2d:bool = True, plot_3d:bool = True, k:int = 3):
    pca = PCA(n_components=pca_component)
    pca_X = pca.fit_transform(X)
    kmeans_X = KMeans(n_clusters=k, random_state=42)
    cluster_labels = kmeans_X.fit_predict(X)
    silhouette_avg = silhouette_score(X, cluster_labels)
    print("Silhouette Score:", silhouette_avg)
    
    if plot_2d:
        plt.scatter(pca_X[:, 0], pca_X[:, 1], c=cluster_labels, cmap='viridis', alpha=0.7)
        plt.xlabel('Principal Component 1')
        plt.ylabel('Principal Component 2')
        plt.colorbar(label='Cluster Label')
        plt.show()
    
    if plot_3d:
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        ax.scatter(pca_X[:, 0], pca_X[:, 1], pca_X[:, 2], c=cluster_labels, cmap='viridis', alpha=0.7)
        ax.set_xlabel('Principal Component 1')
        ax.set_ylabel('Principal Component 2')
        ax.set_zlabel('Principal Component 3')
        plt.show()
    
    if y_true is not None:
        if plot_2d:
            plt.scatter(pca_X[:, 0], pca_X[:, 1], c=y_true, cmap='viridis', alpha=0.7)
            plt.xlabel('Principal Component 1')
            plt.ylabel('Principal Component 2')
            plt.colorbar(label='Cluster Label')
            plt.title('True Label dim = 2')
            plt.show()
            
        if plot_3d:
            fig = plt.figure(figsize=(10, 8))
            ax = fig.add_subplot(111, projection='3d')
            ax.scatter(pca_X[:, 0], pca_X[:, 1], pca_X[:, 2], c=y_true, cmap='viridis', alpha=0.7)
            ax.set_xlabel('Principal Component 1')
            ax.set_ylabel('Principal Component 2')
            ax.set_zlabel('Principal Component 3')
            ax.set_title('True Label dim = 3')
            plt.show()
    
    if y_pred is not None:
        if plot_2d:
            plt.scatter(pca_X[:, 0], pca_X[:, 1], c=y_pred, cmap='viridis', alpha=0.7)
            plt.xlabel('Principal Component 1')
            plt.ylabel('Principal Component 2')
            plt.colorbar(label='Cluster Label')
            plt.title('Predicted Label dim = 2')
            plt.show()
            
        if plot_3d:
            fig = plt.figure(figsize=(10, 8))
            ax = fig.add_subplot(111, projection='3d')
            ax.scatter(pca_X[:, 0], pca_X[:, 1], pca_X[:, 2], c=y_pred, cmap='viridis', alpha=0.7)
            ax.set_xlabel('Principal Component 1')
            ax.set_ylabel('Principal Component 2')
            ax.set_zlabel('Principal Component 3')
            ax.set_title('Predicted Label dim = 3')
            plt.show()

# BoxSim Algorithm Implementation

In [81]:
def distance(x, y):
    return np.sqrt(np.sum((x - y) ** 2))
def get_distances(X, box_ids):
    if box_ids is None or len(box_ids) < 2:
        return [0]
    return [distance(X[i], X[j]) for i, j in combinations(box_ids, 2)]
def to_list(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    return x
def preprocess(X, preprocessing = 'global_mean'):
    match preprocessing:
        case 'global_mean':
            mean = np.mean(np.squeeze(X))
            X_norm = X - mean
            X_bin = (X_norm > 0).astype(int)
        case 'centerize':
            mean = np.mean(X, axis = 0)
            X_norm = X - mean
            X_bin = (X_norm > 0).astype(int)
        case 'normalize': # Actually in this mode it is the same as 'centerize'
            mean = np.mean(X, axis = 0)
            std = np.std(X, axis = 0)
            X_norm = (X - mean) / std
            X_bin = (X_norm > 0).astype(int)
        case 'identity':
            X_bin = (X > 0).astype(int)
        case _:
            X_bin = (X > 0).astype(int)
    return X_bin

In [6]:
def evaluate_ss_within(X, box_ids):
    unique_boxes = np.unique(box_ids)
    ss_within = 0
    for b in unique_boxes:
        box_data = X[box_ids == b]
        if len(box_data) > 1:
            box_mean = np.mean(box_data, axis=0)
            ss_within += np.sum((box_data - box_mean)**2)
    return ss_within

def f_test(X, box_ids, debug=True):
    n_samples = X.shape[0]
    unique_boxes = np.unique(box_ids)
    k = len(unique_boxes)
    
    global_mean = np.mean(X, axis=0)
    ss_total = np.sum((X - global_mean)**2)
    
    if k <= 1:
        print("Warning: All samples are in the same box. F-test is not applicable.")
        return 0, 0, (0, 0), ss_total

    ss_within = evaluate_ss_within(X, box_ids)
            
    ss_between = ss_total - ss_within
    
    df_between = k - 1
    df_within = n_samples - k
    
    ms_between = ss_between / df_between
    ms_within = ss_within / df_within if df_within > 0 else 0
    
    f_value = ms_between / ms_within if ms_within > 0 else float('inf')
    
    p_value = 1 - stats.f.cdf(f_value, df_between, df_within)
    
    if debug:
        print(f"Total Samples (n): {n_samples}")
        print(f"Non-empty Boxes (k): {k}")
        print(f"Degrees of Freedom: df_between={df_between}, df_within={df_within}")
        print(f"Total variance: {ss_total:.4f}")
        print(f"Sum of box variances: {ss_within:.4f}")
        print(f"F-value: {f_value:.4f}")
        print(f"P-value: {p_value:.4e}")
        
    return f_value, p_value, (df_between, df_within), ss_within, ss_total, ms_within

def partial_f_test(X, box_ids_reduced, box_ids_full, debug=True):
    n = X.shape[0]

    ss_within_red = evaluate_ss_within(X, box_ids_reduced)
    ss_within_full = evaluate_ss_within(X, box_ids_full)
    
    df_red = n - len(np.unique(box_ids_reduced))
    df_full = n - len(np.unique(box_ids_full))
    
    # Partial F stat = [(SS_red - SS_full) / (df_red - df_full)] / [SS_full / df_full]
    numerator = (ss_within_red - ss_within_full) / (df_red - df_full)
    denominator = ss_within_full / df_full
    
    f_partial = numerator / denominator if denominator > 0 else 0
    p_value = 1 - stats.f.cdf(f_partial, df_red - df_full, df_full)

    if debug:
        print(f"Total Samples (n): {n}")
        print(f"Degrees of Freedom: df_reduced={df_red}, df_full={df_full}")
        print(f"SS Reduction: {ss_within_red - ss_within_full:.2f}")
        print(f"F-value: {f_partial:.4f}")
        print(f"P-value: {p_value:.4e}")
        
    return f_partial, p_value, (df_red, df_full), ss_within_red, ss_within_full

In [ ]:
def calculate_impurity(X, sub_matrix, method='entropy', is_binary: bool = True, box_ids = None):
    if is_binary:
        combine_dim = sub_matrix.shape[1]
        powers = 2**np.arange(combine_dim)[::-1]
        states = sub_matrix.dot(powers)
        
        _, counts = np.unique(states, return_counts=True)
        p = counts / len(states)
    
    match method, is_binary:
        case 'entropy', True:
            return -np.sum(p * np.log2(p + 1e-9))
        case 'gini', True:
            return 1.0 - np.sum(p**2)
        case 'binary', True:
            p_cols = np.mean(sub_matrix)
            vars = p_cols * (1 - p_cols)
            return np.mean(vars)
        case 'binary_center', True:
            p_cols = np.mean(sub_matrix, axis=0)
            vars = p_cols * (1 - p_cols)
            return np.mean(vars)
        case 'box_variance', False:
            if box_ids is not None:
                return evaluate_ss_within(sub_matrix, box_ids)
            else:
                raise ValueError('box_id should be provided in order to calculate box variance.')
        case 'box_distances', False:
            if box_ids is not None:
                distances = {}
                for i in np.unique(box_ids):
                    distances[i] = get_distances(X, np.arange(len(X))[box_ids == i])
                mean_distances = np.array([np.mean(v) for k, v in distances.items()])
                return mean_distances, distances
            else:
                raise ValueError('box_id should be provided in order to calculate box distances.')
        case 'avg_box_distances', False:
            if box_ids is not None:
                distances = {}
                for i in np.unique(box_ids):
                    distances[i] = get_distances(X, np.arange(len(X))[box_ids == i])
                mean_distances = np.array([np.mean(v) for k, v in distances.items()])
                return np.mean(mean_distances)
            else:
                raise ValueError('box_id should be provided in order to calculate box distances.')
        case _:
            raise ValueError(f"Unknown impurity method: {method}")

In [ ]:
def _evaluate_combination(X_bin, X, cols, combine_dim, impurity):
    sub_bin = X_bin[:, list(cols)]
    
    if impurity in ['box_variance', 'box_distances', 'avg_box_distances']:
        powers = 2**np.arange(combine_dim)[::-1]
        temp_box_ids = sub_bin.dot(powers)
        
        res = calculate_impurity(X, sub_matrix=X, method=impurity, is_binary=False, box_ids=temp_box_ids)
        
        if isinstance(res, tuple):
            return np.mean(res[0])
        return res
    else:
        return calculate_impurity(X, sub_matrix=sub_bin, method=impurity, is_binary=True)

def boxsim_multivariate(X_bin, X=None, ibox=2, combine_dim=2, col_selection='heuristic', 
                        impurity='entropy', reverse=False, debug=False, seed=None,
                        col_selection_k=60, heuristic_samples=3000, plot_hist=False):
    
    if impurity in ['box_variance', 'box_distances'] and X is None:
        raise ValueError(f"Continuous matrix 'X' must be provided for impurity method '{impurity}'.")

    total_features = X_bin.shape[1]
    selected_groups = []
    group_impurities = []
    
    if col_selection == 'random':
        if seed: np.random.seed(seed)
        indices = np.random.choice(total_features, size=ibox * combine_dim, replace=False)
        selected_groups = [tuple(indices[i : i + combine_dim]) for i in range(0, len(indices), combine_dim)]
        group_impurities = [_evaluate_combination(X_bin, X, g, combine_dim, impurity) for g in selected_groups]
        
    elif col_selection == 'order':
        p_cols = np.mean(X_bin, axis=0)
        dist_to_half = np.abs(p_cols - 0.5)
        sorted_cols = np.argsort(dist_to_half)
        if reverse: sorted_cols = sorted_cols[::-1]
        
        indices = sorted_cols[:ibox * combine_dim]
        selected_groups = [tuple(indices[i : i + combine_dim]) for i in range(0, len(indices), combine_dim)]
        group_impurities = [_evaluate_combination(X_bin, X, g, combine_dim, impurity) for g in selected_groups]

    elif col_selection == 'impurity':
        K = min(60, total_features)
        p_cols = np.mean(X_bin, axis=0)
        dist_to_half = np.abs(p_cols - 0.5)
        top_k_cols = np.argsort(dist_to_half)[:K]
        
        all_combs = list(combinations(top_k_cols, combine_dim))
        scored_combs = []
        for comb in all_combs:
            imp = _evaluate_combination(X_bin, X, comb, combine_dim, impurity)
            scored_combs.append((comb, imp))
        
        if impurity in ['box_variance', 'box_distances', 'avg_box_distances']:
            scored_combs.sort(key=lambda x: x[1], reverse=reverse)
        else:
            scored_combs.sort(key=lambda x: x[1], reverse=not reverse)
        
        used_cols = set()
        for comb, imp in scored_combs:
            if not any(c in used_cols for c in comb):
                selected_groups.append(comb)
                group_impurities.append(imp)
                used_cols.update(comb)
            if len(selected_groups) == ibox:
                break
                
        if len(selected_groups) < ibox:
            print(f"Warning: Only found {len(selected_groups)} independent groups.")
    
    elif col_selection == 'heuristic':
        # Pool columns
        if col_selection_k < ibox * combine_dim:
            raise ValueError(f"{col_selection_k =}) should be at least ({ibox * combine_dim =}).")
        total_dims = X_bin.shape[1]
        all_single_scores = []
        for d in range(total_dims):
            score = _evaluate_combination(X_bin, X, [d], 1, impurity)
            all_single_scores.append((d, score))
        if impurity in ['box_variance', 'box_distances', 'avg_box_distances']:
            all_single_scores.sort(key=lambda x: x[1], reverse=reverse)
        else:
            all_single_scores.sort(key=lambda x: x[1], reverse=not reverse)
        seed_pool = [x[0] for x in all_single_scores[:col_selection_k]]
        
        # Random Greedy Search
        selected_groups = []
        group_impurities = []
        while len(selected_groups) < ibox and len(seed_pool) >= combine_dim:
            all_possible_combs = list(combinations(seed_pool, combine_dim))
            if len(all_possible_combs) > heuristic_samples:
                sample_indices = np.random.choice(len(all_possible_combs), heuristic_samples, replace=False)
                candidate_combs = [all_possible_combs[i] for i in sample_indices]
            else:
                candidate_combs = all_possible_combs
            
            current_round_scored = []
            for comb in candidate_combs:
                imp = _evaluate_combination(X_bin, X, comb, combine_dim, impurity)
                current_round_scored.append((comb, imp))
            
            if impurity in ['box_variance', 'box_distances', 'avg_box_distances']:
                best_match = min(current_round_scored, key=lambda x: x[1]) if not reverse else max(current_round_scored, key=lambda x: x[1])
            else:
                best_match = max(current_round_scored, key=lambda x: x[1]) if not reverse else min(current_round_scored, key=lambda x: x[1])
            
            best_comb, best_imp = best_match
            selected_groups.append(best_comb)
            group_impurities.append(best_imp)
            
            for col in best_comb:
                seed_pool.remove(col)
            
            if plot_hist:
                plt.figure(figsize=(8, 6))
                plt.hist([imp for _, imp in current_round_scored], bins=30)
                plt.vlines(best_imp, 0, col_selection_k, colors='red')
                plt.show()
    
    grouped_cols = [[int(col) for col in group] for group in selected_groups]
    flattened_cols = [int(col) for group in selected_groups for col in group]
    sub_matrix = X_bin[:, flattened_cols]
    
    total_dims = len(flattened_cols)
    num_possible = 2**total_dims
    powers = 2**np.arange(total_dims)[::-1]
    box_ids = sub_matrix.dot(powers)
    counts = np.bincount(box_ids, minlength=num_possible)
    
    if debug:
        print(f"Multivariate mode using column selection method ({col_selection})")
        for i, (group, imp) in enumerate(zip(selected_groups, group_impurities)):
            print(f"Group {i+1}: Columns {[int(i) for i in group]} | {impurity.capitalize()}: {imp:.4f}")
        print(f"Flattened Dimensions used for Box ID: {grouped_cols}")
        print(f"Total Possible Boxes: {num_possible}")
        print(f"Non-empty Boxes: {np.count_nonzero(counts)}")
        
    return box_ids, sub_matrix, num_possible, counts, selected_groups

In [9]:
def plot_binary_boxes(y_true, ibox:int = 5, num_possible:int = 32, counts = None, box_ids = None,
                      shape:tuple[2] = None, fig_size:tuple[2] = None,
                      dpi:int = 80, cmap = 'white', show_idx:int = 0, idx_per_line:int = 4,
                      idx_size:int = 6,  show_num:int = 0, show_pie:int = 0, 
                      edge_color = 'w', jitter:float = 0.02,
                      point_size:int = 6, edge_size:int = 0, show_points:int = 0,
                      point_color = 'blue', bin_strs = None
                    ):
    combined_labels = []
    
    for i in range(num_possible):
        if bin_strs is None:
            bin_str = bin(i)[2:].zfill(ibox)
        else:
            bin_str = bin_strs[i]
        if counts[i] > show_num:
            combined_labels.append(f"{bin_str}\n{counts[i]}\n")
        else:
            combined_labels.append(f"{bin_str}\n\n")
    pad_size = (shape[0] * shape[1] - num_possible)
    if not isinstance(pad_size, int) and not isinstance(pad_size, float):
        pad_size = pad_size.astype(np.int32)
    else:
        pad_size = int(pad_size)
    if pad_size < 0:
        raise ValueError("shape is not enough to show all boxes")
    
    W, H = shape[0], shape[1]
        
    grid_data = np.pad(counts, (0, pad_size)).reshape(W, H).T
    grid_labels = np.array(combined_labels + [""] * pad_size).reshape(W, H).T
    
    plt.figure(figsize = fig_size, dpi = dpi)    
    ax = sns.heatmap(grid_data, annot=grid_labels, fmt="", cmap=mcolors.ListedColormap([cmap]),
                    square=True, cbar=False, linewidths=.6, linecolor="black")
    
    if y_true is not None:
        unique_classes = np.unique(y_true)
        color_map = plt.get_cmap('tab10')

        for i in range(num_possible):
            row_idx = i % H 
            col_idx = i // H 
            
            in_box_mask = (box_ids == i)
            total_in_box = counts[i]
            
            if total_in_box == 0:
                continue
                
            if total_in_box > show_pie:
                class_counts = [np.sum(y_true[in_box_mask] == cls) for cls in unique_classes]
                sizes = [c for c in class_counts if c > 0]
                colors = [color_map(idx) for idx, c in enumerate(class_counts) if c > 0]
                
                radius = 0.2 + (total_in_box / counts.max()) * 0.28
                
                labels = sizes
                start_angle = 90
                for size, color in zip(sizes, colors):
                    angle = 360 * (size / sum(sizes))
                    wedge = plt.matplotlib.patches.Wedge(
                        center=(col_idx + 0.5, row_idx + 0.5), r=radius, 
                        theta1=start_angle, theta2=start_angle + angle, 
                        facecolor=color, edgecolor=edge_color, linewidth=0.5, zorder=3
                    )
                    ax.add_patch(wedge)
                    start_angle += angle
                # print("pied ", i)
                
            elif total_in_box > show_points and show_points > 0:
                for class_idx, cls in enumerate(unique_classes):
                    cls_in_box_mask = (y_true == cls) & in_box_mask
                    if not np.any(cls_in_box_mask): continue
                    num_pts = np.sum(cls_in_box_mask)
                    jitter_x = np.random.uniform(jitter, 1 - jitter, size=num_pts)
                    jitter_y = np.random.uniform(jitter, 1 - jitter, size=num_pts)
                    ax.scatter(col_idx + jitter_x, row_idx + jitter_y, 
                            color=color_map(class_idx), s=point_size,
                            alpha=0.8, edgecolors=edge_color, linewidth=edge_size, zorder=2)

        legend_elements = [Line2D([0], [0], marker='o', color='w', label=f'Class {cls}',
                        markerfacecolor=color_map(i), markersize=8) for i, cls in enumerate(unique_classes)]
        ax.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left', title="True Classes")
    elif show_points > 0:
        w_coords = box_ids // H
        h_coords = box_ids % H
        jitter_x = np.random.uniform(jitter, 1 - jitter, size=len(box_ids))
        jitter_y = np.random.uniform(jitter, 1 - jitter, size=len(box_ids))
        scatter_x = w_coords + jitter_x
        scatter_y = h_coords + jitter_y
        ax.scatter(scatter_x, scatter_y, color=point_color, s=point_size,
                    alpha=0.7, edgecolors=edge_color, linewidth=edge_size, zorder=2)
        
    for i in range(num_possible):
        if 0 < counts[i] <= show_idx:
            idx = np.where(box_ids == i)[0]
            lines = []
            for j in range(0, len(idx), idx_per_line):
                line_parts = idx[j : j + idx_per_line]
                lines.append("  ".join(map(str, line_parts)))
            idx_str = "\n".join(lines)
            plt.text(i // H + 0.5, i % H + 0.666, idx_str, fontsize = idx_size, ha = 'center', va = 'center')
    
    plt.tight_layout()
    plt.show()
    return

In [106]:
def boxsim(X, y_true = None, cols = None,
        var_threshold: tuple[2] = (-1, 2), ibox: int = 5, 
        col_selection = 'order', col_selection_k: int = 60, heuristic_samples: int = 3000,
        debug: bool = False, reverse: bool = False, mode: str = 'binary', seed: int = None,
        preprocessing: str = 'global_mean', pca_k: int = 0,
        combine_dim: int = 1, impurity: str = 'entropy',
        shape:tuple[2] = None, fig_size:tuple[2] = None, fig_default_shape:bool = True,
        dpi:int = 80, cmap = 'white', show_idx:int = 0, idx_per_line:int = 4,
        idx_size:int = 6,  show_num:int = 0, show_pie:int = 0, 
        edge_color = 'w', jitter:float = 0.02, 
        point_size:int = 6, edge_size:int = 0, show_points:int = 0, 
        point_color = 'blue', no_plot: bool = False, plot_hist: bool = False,
        test = None, report_test: bool = False):
    """The function clusters the input dataframe and plots the result if needed.

    Args:
        X (numpy.ndarray or matrices): Input dataframe.
        y_true (numpy.ndarray or vectors, optional): True data classes, usually in the form of integer vectors. Defaults to None.
        cols (list, optional): Manually chosen dimensions (columns) of X, used only for plot and works under mode 'selected'. Defaults to None.
        var_threshold (tuple[2], optional): A pair of lower and upper bound, used to rule out columns having Hamming variance calculated columnwise outside of the thresholds. Defaults to (-1, 2).
        ibox (int, optional): Determines how many columns or combinations of columns are used in box clustering. Defaults to 5.
        col_selection (str, optional): Method to select the columns to be combined, available in 'multivariate' mode. Defaults to 'order'.
        col_selection_k (int, optional): Determines how many columns to be considered in the pool as following `col_selection`. Defaults to 60.
        heuristic_samples (int, optional): Determines how many sampling shall be made when heuristically choosing columns. Defaults to 3000.
        debug (bool, optional): Set to true to output box clustering important information. Defaults to False.
        reverse (bool, optional): In modes 'binary' and 'multivariate' and col_selection 'order', determines which columns to choose first, True means choosing from columns of maximal variance to columns of minimal variance. Defaults to False.
        mode (str, optional): Different modes of box clustering, 'binary', 'random', 'multivariate' or 'selected'. Defaults to 'binary'.
        seed (int, optional): Random seed used to randomly choose columns, fixed if provided, only works under mode 'random'. Defaults to None.
        preprocessing (str, optional): Determines how data X is preprocessed, 'global_mean', 'centerize', 'normalize' or 'identity'. Defaults to 'global_mean'.
        pca_k (int, optional): Determines how many PCA components are used, set to 0 when no PCA is required to be pre-performed to X. Defaults to 0.
        combine_dim (int, optional): Determines how many columns are to be combined in mode 'multivariate'. Defaults to 1.
        impurity (str, optional): Determines which impurity metric is used as a substitute of variance, 'entropy', 'gini' or 'variance'. Defaults to 'entropy'.
        shape (tuple[2], optional): Determines in which shape the boxes are to be illustrated, automatically determined from `ibox`, etc. if not preset. Defaults to None.
        fig_size (tuple[2], optional): Determines in which size the plot shall be presented as in `matplotlib.pyplot`. Defaults to None.
        fig_default_shape (bool, optional): Use automatically determined fig_size if set to True. Defaults to True.
        dpi (int, optional): The dpi of the plot. Defaults to 80.
        cmap (str, optional): The color map of the impurity metric in the plot. Defaults to 'white'.
        show_idx (int, optional): Displays the row id of X in each box if set to positive. Defaults to 0.
        idx_per_line (int, optional): Determines how many ids are grouped in one line when displayed. Defaults to 4.
        idx_size (int, optional): The size of displayed id texts. Defaults to 6.
        show_num (int, optional): Displays the numeric form of (maximum of) how many data rows are clustered into each box in the plot. Defaults to 0.
        show_pie (int, optional): Displays the distribution of rows in each box in the plot if classes were provided in `y_true` whenever the box contains greater than or equal amount to this integer parameter. Defaults to 0.
        edge_color (str, optional): Box frame color in the plot. Defaults to 'w'.
        jitter (float, optional): If data were presented as points, all points in the same box are randomly distributed in the box with (x, y) uniformly sampled from the interval (jitter, 1-jitter). Defaults to 0.02.
        point_size (int, optional): Point size in the plot. Defaults to 6.
        edge_size (int, optional): Thickness of box frames. Defaults to 0.
        show_points (int, optional): Determines (maximum of) how many data rows are displayed as points in each box in the plot. Defaults to 0.
        point_color (str, optional): The default point color if no `y_true` was provided. Defaults to 'blue'.
        no_plot (bool, optional): Set to true if no plot is needed. Defaults to False.
        plot_hist (bool, optional): Set to true if histograms of impurity in heuristic choice of columns are needed. Defaults to False.
        test (str, optional): Perform a statistical test when provided, 'F' or 'distances'. Defaults to None.
        report_test (bool, optional): Set to true if the test results and details are needed. Defaults to False.

    Raises:
        ValueError: Usually raised when `ibox` is conflicted to `shape`.

    Returns:
        box_ids, col_idx, test_results: The `box_ids` contains the box ids of each data row as clustered. The `col_idx` are a list of chosen columns or dimensions' ids. The `text_results` are crucial results provided by the statistical test. 
    """

    # Decide shape and fig_size
    total_features_estimate = 2 ** (combine_dim * ibox) if mode == 'multivariate' else 2 ** ibox
    
    if shape is None:
        shape = (np.ceil(np.sqrt(total_features_estimate)).astype(np.int32),
                 np.ceil(np.sqrt(total_features_estimate)).astype(np.int32))
    
    if fig_default_shape:
        fig_size = (np.ceil(shape[0]/shape[1] * 6), 6)
    elif fig_size is None:
            fig_size = (max(np.ceil(np.sqrt(total_features_estimate)), 4),
                        max(np.ceil(np.sqrt(total_features_estimate)), 4))
    
    # PCA
    if pca_k > 0:
        pca = PCA(n_components = pca_k)
        X = pca.fit_transform(X)
        if debug: print(f"PCA applied: {X.shape}")
    
    # Preprocessing
    if mode in ['binary', 'random', 'selected', 'multivariate']:
        X_bin = preprocess(X, preprocessing)

    elif mode in ['other']:
        pass
    
    # Binary
    if mode == 'binary':
        col_vars = np.count_nonzero(X_bin, axis = 0) / X_bin.shape[0]
        col_mask = col_vars > 0.5
        col_vars[col_mask] = 1 - col_vars[col_mask]
        
        valid_col_mask = np.logical_and(col_vars > var_threshold[0], col_vars < var_threshold[1])
        masked_col_vars = col_vars[valid_col_mask]
        X_mid = X_bin[:, valid_col_mask]
        sorted_col_idx = np.lexsort((masked_col_vars,))
        if reverse: sorted_col_idx = np.lexsort((-masked_col_vars,))
        X_col_sorted = X_mid[:, sorted_col_idx]
        
        num_possible = 2**ibox
        sub_matrix = X_col_sorted[:, :ibox]
        powers = 2**np.arange(ibox)[::-1]
        box_ids = sub_matrix.dot(powers)
        counts = np.bincount(box_ids, minlength=num_possible)
        
        col_idx = to_list(sorted_col_idx)
    
        if debug:
            print("Parameters: random, seed, combine_dim, col_selection, impurity are neglected.")
            print("Column Variances:", col_vars)
            print("Min Variance:", masked_col_vars[sorted_col_idx][0])
            print("Max Variance:", masked_col_vars[sorted_col_idx][-1])
            print("Sorted Column Indices:", sorted_col_idx)
            print(dict(zip(col_idx, to_list(masked_col_vars[sorted_col_idx]))))
            print("Box Counts:", counts)
            
    if mode == 'selected':
        ibox = len(cols)
        col_vars = np.count_nonzero(X_bin, axis = 0) / X_bin.shape[0]
        col_mask = col_vars > 0.5
        col_vars[col_mask] = 1 - col_vars[col_mask]
        masked_col_vars = col_vars[cols]
        X_col_sorted = X_bin[:, cols]
        num_possible = 2**ibox
        sub_matrix = X_col_sorted[:, :ibox]
        powers = 2**np.arange(ibox)[::-1]
        box_ids = sub_matrix.dot(powers)
        counts = np.bincount(box_ids, minlength=num_possible)
        
        col_idx = to_list(cols)
        masked_col_vars = to_list(masked_col_vars)
        
        if debug:
            print("Parameters: ibox, random, seed, combine_dim, col_selection, impurity are neglected.")
            print("Column Variances:", col_vars)
            print("Min Variance:", np.min(col_vars))
            print("Max Variance:", np.max(col_vars))
            print("Chosen Column Variences:", dict(zip(cols, masked_col_vars)))
            print("Box Counts:", counts)
    
    if mode == 'random':
        total_features = X_bin.shape[1]
        if ibox > total_features:
            raise ValueError(f"ibox ({ibox}) cannot exceed #features ({total_features})")
        if seed: np.random.seed(seed)
        random_col_idx = np.random.choice(total_features, size=ibox, replace=False)
        
        sub_matrix = X_bin[:, random_col_idx]
        num_possible = 2**ibox
        powers = 2**np.arange(ibox)[::-1]
        box_ids = sub_matrix.dot(powers)
        counts = np.bincount(box_ids, minlength=num_possible)
        
        col_idx = random_col_idx
        
        if debug:
            print(f"Randomly Selected Column Indices: {random_col_idx}")
            col_vars = np.count_nonzero(sub_matrix, axis=0) / sub_matrix.shape[0]
            print(f"Variances: {to_list(col_vars)}")
            print(f"Box Counts:", counts)
    
    if mode == 'multivariate':
        box_ids, sub_matrix, num_possible, counts, col_idx = boxsim_multivariate( \
            X_bin, X, ibox, combine_dim, col_selection, impurity, reverse, debug, seed,
            col_selection_k, heuristic_samples, plot_hist)

    
    if mode in ['binary', 'random'] and not no_plot:
        plot_binary_boxes(y_true, ibox, num_possible, counts, box_ids, shape, fig_size, \
                      dpi, cmap, show_idx, idx_per_line, idx_size, show_num, show_pie, \
                      edge_color, jitter, point_size, edge_size, show_points, point_color)
    
    if mode == 'selected' and not no_plot:
        plot_binary_boxes(y_true, ibox, num_possible, counts, box_ids, shape, fig_size, \
                      dpi, cmap, show_idx, idx_per_line, idx_size, show_num, show_pie, \
                      edge_color, jitter, point_size, edge_size, show_points, point_color)
        
    if mode in ['multivariate'] and not no_plot:
        plot_binary_boxes(y_true, ibox, num_possible, counts, box_ids, shape, fig_size, \
                      dpi, cmap, show_idx, idx_per_line, idx_size, show_num, show_pie, \
                      edge_color, jitter, point_size, edge_size, show_points, point_color)

    if not test:
        return np.array(box_ids), col_idx, None
    else:
        match test:
            case 'F':
                test_results = f_test(X, box_ids, report_test)
                return np.array(box_ids), col_idx, test_results
            case 'distances':
                distances = {}
                for i in np.unique(box_ids):
                    distances[i] = get_distances(X, np.arange(len(X))[box_ids == i])
                return np.array(box_ids), col_idx, distances
            case _:
                return np.array(box_ids), col_idx, None